CommonvoiceのJsonファイルを作成する

In [1]:
import os
import random
import numpy as np
import soundfile as sf
import uuid
import tqdm
import soundfile as sf
import numpy as np
import csv
from tqdm import tqdm
import math

model = "ja"
database_folder = "./downloaded_folder"
corpus_database_folder = database_folder + f"/commonvoice/{model}"

In [2]:
import shutil
if not os.path.exists(corpus_database_folder):
        os.makedirs(corpus_database_folder)
        print(f"Created folder: {corpus_database_folder}")
else:
    print(f"Folder already exists: {corpus_database_folder}")
shutil.rmtree(corpus_database_folder, ignore_errors=True)

from datasets import load_dataset
ds = load_dataset(f"mozilla-foundation/common_voice_11_0", "ja", trust_remote_code=True, cache_dir=corpus_database_folder)

Folder already exists: ./downloaded_folder/commonvoice/ja


/home/tsukagoshitoshihiro/.pyenv/versions/apsipa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Reading metadata...: 6505it [00:00, 191385.71it/s]les/s]
Generating train split: 6505 examples [00:00, 11220.49 examples/s]
Reading metadata...: 4485it [00:00, 414769.45it/s]examples/s]
Generating validation split: 4485 examples [00:00, 12133.47 examples/s]
Reading metadata...: 4604it [00:00, 398041.30it/s]es/s]
Generating test split: 4604 examples [00:00, 12038.03 examples/s]
Reading metadata...: 1146it [00:00, 331700.53it/s]les/s]
Generating other split: 1146 examples [00:00, 12045.83 examples/s]
Reading metadata...: 2464it [00:00, 407104.90it/s] examples/s]
Generating invalidated split: 2464 examples [00:00, 12103.28 examples/s]


In [3]:
file_path = ds["train"][0]["audio"]["path"]

# 音声ファイルを読み込み
waveform, sample_rate = sf.read(file_path)
sample_rate

32000

In [4]:
import resampy
import numpy as np
import soundfile as sf
from tqdm import tqdm

# 必要なサンプリングレート
target_sample_rate = 16000

# データセットのファイルを順次処理
for i in tqdm(range(len(ds["train"])), desc="Resampling and overwriting audio files"):
    # ファイルパスを取得
    file_path = ds["train"][i]["audio"]["path"]
    
    # 音声ファイルを読み込み
    waveform, sample_rate = sf.read(file_path)
    
    # サンプリングレートが異なる場合、リサンプリングを実行
    if sample_rate != target_sample_rate:
        waveform = resampy.resample(waveform, sr_orig=sample_rate, sr_new=target_sample_rate)
        
        # リサンプリングした波形を元のファイルに上書き保存
        sf.write(file_path, waveform, target_sample_rate)




# サンプリングレート変更後の確認
print(f"Resampling completed. Files are now saved with {target_sample_rate} Hz.")





# データセットのファイルを順次処理
for i in tqdm(range(len(ds["validation"])), desc="Resampling and overwriting audio files"):
    # ファイルパスを取得
    file_path = ds["validation"][i]["audio"]["path"]
    
    # 音声ファイルを読み込み
    waveform, sample_rate = sf.read(file_path)
    
    # サンプリングレートが異なる場合、リサンプリングを実行
    if sample_rate != target_sample_rate:
        waveform = resampy.resample(waveform, sr_orig=sample_rate, sr_new=target_sample_rate)
        
        # リサンプリングした波形を元のファイルに上書き保存
        sf.write(file_path, waveform, target_sample_rate)


# データセットのファイルを順次処理
for i in tqdm(range(len(ds["test"])), desc="Resampling and overwriting audio files"):
    # ファイルパスを取得
    file_path = ds["test"][i]["audio"]["path"]
    
    # 音声ファイルを読み込み
    waveform, sample_rate = sf.read(file_path)
    
    # サンプリングレートが異なる場合、リサンプリングを実行
    if sample_rate != target_sample_rate:
        waveform = resampy.resample(waveform, sr_orig=sample_rate, sr_new=target_sample_rate)
        
        # リサンプリングした波形を元のファイルに上書き保存
        sf.write(file_path, waveform, target_sample_rate)


Resampling and overwriting audio files: 100%|██████████| 6505/6505 [04:03<00:00, 26.67it/s]


Resampling completed. Files are now saved with 16000 Hz.


Resampling and overwriting audio files: 100%|██████████| 4604/4604 [03:06<00:00, 24.68it/s]


In [5]:
file_path = ds["train"][0]["audio"]["path"]

# 音声ファイルを読み込み
waveform, sample_rate = sf.read(file_path)
sample_rate

16000

In [6]:
import json
import soundfile as sf
from tqdm import tqdm
import webrtcvad

# =============================
# WebRTC VAD の初期化
# =============================
vad = webrtcvad.Vad()
vad.set_mode(3)  # 0: 高感度, 3: 低感度

# =============================
# テキスト前処理関数
# =============================
def preprocess_transcript(transcript):
    # 指定された文字を削除
    chars_to_remove = ['！', '!', '“', '”', '’', '"', "'", '，', '。', '、', '「', '」', '・', ',']
    for char in chars_to_remove:
        transcript = transcript.replace(char, "")
    
    # 一文字ごとに空白を入れる
    spaced_transcript = " ".join(transcript)
    
    return spaced_transcript

# =============================
# WebRTC VAD を使用した発話区間検出
# =============================
def detect_speech(audio_path, sample_rate=16000, frame_duration_ms=30):
    """
    WebRTC VAD を使用して発話区間を検出します。
    """
    audio, sr = sf.read(audio_path)
    if sr != sample_rate:
        raise ValueError(f"Sample rate must be {sample_rate}, but got {sr}")

    frame_size = int(sample_rate * frame_duration_ms / 1000)
    audio = (audio * 32768).astype("int16")  # WebRTC VAD は 16-bit PCM を要求

    speech_timestamps = []
    is_speech = False
    speech_start = 0

    for i in range(0, len(audio), frame_size):
        frame = audio[i:i+frame_size]
        if len(frame) < frame_size:
            break

        is_speech_frame = vad.is_speech(frame.tobytes(), sample_rate)
        if is_speech_frame and not is_speech:
            is_speech = True
            speech_start = i / sample_rate
        elif not is_speech_frame and is_speech:
            is_speech = False
            speech_end = i / sample_rate
            speech_timestamps.append([speech_start, speech_end])

    # 発話が終わっていない場合の処理
    if is_speech:
        speech_timestamps.append([speech_start, len(audio) / sample_rate])

    return speech_timestamps

# =============================
# JSON 出力用のリスト
# =============================
all_data_for_json = []

# =============================
# データセットの処理
# =============================
def process_dataset(dataset_name, dataset, start_index):
    audio_paths = []
    cv_data = []
    index_offset = start_index

    for i in tqdm(range(len(dataset)), desc=f"Processing {dataset_name} data"):
        try:
            audio_path = dataset[i]["audio"]["path"]
            # 音声ファイルの読み込み
            waveform, sample_rate = sf.read(audio_path)
            # 音声ファイルの長さを計算
            duration = len(waveform) / sample_rate
            # 10秒未満のファイルのみをリストに追加
            if duration < 10:
                audio_paths.append(audio_path)
                audio_transcript = dataset[i]["sentence"]
                processed_transcript = preprocess_transcript(audio_transcript)
                
                cv_data.append([index_offset + i, audio_path, processed_transcript])
                
                # WebRTC VAD を使用した発話区間検出
                speech_timestamps = detect_speech(audio_path)
                
                # タイムスタンプ辞書
                timestamps = {
                    "chewing": [],
                    "swallowing": [],
                    "noise": [],
                    "speech": speech_timestamps,
                    "mask": []
                }
                
                # JSON 用の辞書データを作成
                record = {
                    "path": audio_path,
                    "timestamps": timestamps,
                    "text": processed_transcript
                }
                all_data_for_json.append(record)
                
        except Exception as e:
            print(f"Error processing {dataset_name} file {i}: {e}")

    return len(audio_paths), cv_data, index_offset + len(dataset)

# =============================
# train, validation, test セットの処理
# =============================
train_count, train_cv_data, next_index = process_dataset("train", ds["train"], 0)
valid_count, valid_cv_data, next_index = process_dataset("validation", ds["validation"], next_index)
test_count, _, _ = process_dataset("test", ds["test"], next_index)

# =============================
# JSON ファイル出力
# =============================
output_file = "./../json/commonvoice.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_data_for_json, f, ensure_ascii=False, indent=4)

print(f"Filtered audio paths (train): {train_count} files")
print(f"Filtered audio paths (validation): {valid_count} files")
print(f"Filtered audio paths (test): {test_count} files")
print(f"JSON file has been saved as {output_file}")


Processing train data:   0%|          | 0/6505 [00:00<?, ?it/s]

Processing test data: 100%|██████████| 4604/4604 [00:24<00:00, 184.34it/s]


Filtered audio paths (train): 6494 files
Filtered audio paths (validation): 4433 files
Filtered audio paths (test): 4528 files
JSON file has been saved as ./../json/commonvoice.json
